# Module 6 — Decompose with Codex
## Wildfire data and dashboards

Download before class. Inspect, clean, plan, and build in class.

- Source: [USDA FPA FOD, seventh edition](https://doi.org/10.2737/RDS-2013-0009.7), 1992–2024.
- Class scope: **California**. One row is a reported wildfire occurrence.
- Keep this notebook in the repository root. Use this folder's `.venv` kernel.
- Use Codex for short hints and feedback on your attempt; you write the code.

## Before class
Run `uv sync --frozen` in Terminal or PowerShell once. Then run the cell below.

`download_data()` downloads the source and exports the full nationwide table to
`wildfire_raw.csv`. It returns the CSV path. Completed files are reused.
The CSV keeps every record and column. No cleaning or filtering happens here.

In [1]:
from pathlib import Path
import pandas as pd
from download_wildfire import download_data

source = download_data()

Source already available: /Users/yeganehalimohammadi/dso576/06-decompose/data/raw/fpa_fod_7th/S_USA.Fire_FPA_FOD_7th_Fires.gdb
Raw CSV already available: /Users/yeganehalimohammadi/dso576/06-decompose/data/raw/wildfire_raw.csv


## Read in chunks

Read the national CSV 100,000 rows at a time. This avoids holding the full table
in memory. All columns are read as text, preserving codes with leading zeros.

- `keep_default_na=False`: keep strings such as `"NA"` as text.
- `na_values=[""]`: treat only empty fields as missing.
- Choose numeric and date types during class.

See [data/README.md](data/README.md) for the column definitions.

In [2]:
chunks = pd.read_csv(
    source,
    chunksize=100_000,
    dtype="string",
    keep_default_na=False,
    na_values=[""],
)

type(chunks)

pandas.io.parsers.readers.TextFileReader

## Keep California from each chunk

The `for` loop reads one chunk, keeps its CA rows, and appends them to `parts`.
After the loop, `pd.concat()` stacks those rows into one DataFrame. All columns
are kept. This selects the class scope; it does not clean the values.
Then `del chunk, parts` removes the temporary variables; `ca` stays available.

| Variable | What it holds |
| --- | --- |
| `chunks` | An iterator that reads the next chunk when the loop asks for it; not a list |
| `chunk` | The current DataFrame, with up to 100,000 rows |
| `parts` | A list of DataFrames containing CA rows |
| `ca` | One DataFrame containing all CA rows |

**To run the loop again, rerun the cell that creates `chunks` first.**
An iterator is used up after one pass.

In [3]:
parts = []

for chunk in chunks:
    parts.append(chunk.loc[chunk["STATE"] == "CA"])

len(parts)

27

In [4]:
ca = pd.concat(parts, ignore_index=True)

del chunk, parts

## 1 Inspect — in class

Inspect `ca`: columns, types, missing values, categories, and record identifiers.
Explain what one row represents. Use the handout to guide your work.

In [5]:
ca.head()

,OBJECTID,FOD_ID,FPA_ID,SOURCE_SYSTEM_TYPE,SOURCE_SYSTEM,NWCG_REPORTING_AGENCY,NWCG_REPORTING_UNIT_ID,NWCG_REPORTING_UNIT_NAME,SOURCE_REPORTING_UNIT,SOURCE_REPORTING_UNIT_NAME,...,FIRE_SIZE,FIRE_SIZE_CLASS,LATITUDE,LONGITUDE,OWNER_DESCR,STATE,COUNTY,FIPS_CODE,FIPS_NAME,GLOBALID
0,8075,135,FS-1419180,FED,FS-FIRESTAT,FS,USNVHTF,Humboldt-Toiyabe National Forest,0417,Humboldt-Toiyabe National Forest,...,0.5,B,38.83972222,-119.88638889,USFS,CA,3,06003,Alpine County,{A0DC6C66-D4D0-4249-BB0C-962F7E3F0899}
1,8089,358,FS-1419642,FED,FS-FIRESTAT,FS,USCAANF,Angeles National Forest,0501,Angeles National Forest,...,0.1,A,34.16222222,-117.68166667,MISSING/NOT SPECIFIED,CA,<NA>,<NA>,<NA>,{0C57802B-C42F-4AD0-B44A-A5656C3F3099}
2,8096,562,FS-1420092,FED,FS-FIRESTAT,FS,USCASQF,Sequoia National Forest,0513,Sequoia National Forest,...,0.1,A,36.04472222,-118.15722222,USFS,CA,<NA>,<NA>,<NA>,{FDC01D1C-A922-4C34-9183-0FFD5AD589B8}
3,8100,566,FS-1420114,FED,FS-FIRESTAT,FS,USCAPNF,Plumas National Forest,0511,Plumas National Forest,...,0.2,A,39.57972222,-121.12527778,USFS,CA,7,06007,Butte County,{280E42B4-F7F2-46F9-9C5A-1685B447B23E}
4,8132,150,FS-1419212,FED,FS-FIRESTAT,FS,USCASQF,Sequoia National Forest,0513,Sequoia National Forest,...,0.1,A,35.76888889,-118.43527778,STATE OR PRIVATE,CA,<NA>,<NA>,<NA>,{5D12AA26-5086-46A5-80F5-D78191FEB9F0}


## 2 Clean — in class

Record your proposed rules in a short Markdown cell. Keep `ca` unchanged.
Create your analysis table in a new variable and save it to `outputs/wildfire_clean.csv`.
Add cells as needed; use the handout for the required evidence.

### Validate the prepared California table

Run these three checks after the cells that create `ca`. They read the prepared table from `outputs/` and leave `ca` unchanged. A passing check supports the stated cleaning rule; it does not prove that every source report is accurate.

**Check 1 — Were records and reported acres preserved?** Compare the cleaned table with the original California rows. Every `FOD_ID` should remain in the same order, and acreage should agree within floating-point tolerance.

In [ ]:
# Load the cleaned table;
fires_clean = pd.read_parquet("outputs/fires_clean.parquet")

# Convert the original acreage text to numbers before comparing totals.
# ca is the raw data of california fires.
source_acres = pd.to_numeric(ca["FIRE_SIZE"], errors="raise").sum()
clean_acres = fires_clean["FIRE_SIZE"].sum()

# Reset the indexes so this checks ID values and row order, not index labels.
same_ids = ca["FOD_ID"].reset_index(drop=True).equals(
    fires_clean["FOD_ID"].reset_index(drop=True)
)
# Summarize the preservation checks; a difference under 0.000001 acre is rounding noise.
record_check = pd.DataFrame({
    "test": ["Rows and columns", "Unique FOD_ID values", "Same FOD_ID order", "Acreage unchanged"],
    "passed": [
        ca.shape == fires_clean.shape and ca.columns.equals(fires_clean.columns),
        fires_clean["FOD_ID"].notna().all() and fires_clean["FOD_ID"].is_unique,
        same_ids,
        abs(source_acres - clean_acres) < 1e-6,
    ],
})
# Show the totals and a pass/fail result for each rule.
print(f"Rows: {len(fires_clean):,} | Columns: {fires_clean.shape[1]}")
print(f"Source acres: {source_acres:,.4f} | Clean acres: {clean_acres:,.4f}")
record_check

Rows: 283,285 | Columns: 39
Source acres: 25,622,806.7449 | Clean acres: 25,622,806.7449


,test,passed
0,Rows and columns,True
1,Unique FOD_ID values,True
2,Same FOD_ID order,True
3,Acreage unchanged,True


**Check 2 — Do county totals reconcile?** Include records without a reported county as their own group. These records still contribute to the statewide acreage total.

In [8]:
# Use the source FIPS code for grouping and keep missing counties visible.
county_acres = (
    fires_clean.assign(
        county_group=(
            fires_clean["FIPS_CODE"].fillna("Unreported")
            + " | "
            + fires_clean["FIPS_NAME"].fillna("County not reported")
        )
    )
    .groupby("county_group")["FIRE_SIZE"]
    .sum()
    .sort_values(ascending=False)
)
# Summing every county group should recover the statewide acreage total.
county_total = county_acres.sum()
print(f"County groups: {len(county_acres)}")
print(f"County total: {county_total:,.4f} acres")
print(f"Statewide total: {clean_acres:,.4f} acres")
print(f"Reconciles: {abs(county_total - clean_acres) < 1e-6}")
# Show how much acreage has no reported county, then the largest groups.
print(f"County not reported: {county_acres['Unreported | County not reported']:,.4f} acres")
county_acres.head(5).rename("reported_acres").to_frame()

County groups: 59
County total: 25,622,806.7449 acres
Statewide total: 25,622,806.7449 acres
Reconciles: True
County not reported: 6,314,947.8920 acres


,reported_acres
county_group,
Unreported | County not reported,6314947.892
06063 | Plumas County,1718307.12
06093 | Siskiyou County,1682449.22
06105 | Trinity County,1214359.998
06019 | Fresno County,814226.092


**Check 3 — Are calendar fields consistent?** The discovery year and day of year should match the discovery date. Where containment is reported, its day of year should match its own date, and its calendar date should not precede discovery.

In [9]:
# These are parsed dates in the prepared table.
discovery = fires_clean["DISCOVERY_DATE"]
containment = fires_clean["CONT_DATE"]
# Count rows where stored year/day fields disagree with their dates.
# Missing containment values are checked as a pair, and dates are compared by calendar day.
date_check = pd.DataFrame({
    "test": [
        "Discovery year disagrees",
        "Discovery day of year disagrees",
        "Containment day of year disagrees",
        "Containment date and day of year missing separately",
        "Containment date precedes discovery",
    ],
    "problem_rows": [
        discovery.dt.year.ne(fires_clean["FIRE_YEAR"]).fillna(False).sum(),
        discovery.dt.dayofyear.ne(fires_clean["DISCOVERY_DOY"]).fillna(False).sum(),
        containment.dt.dayofyear.ne(fires_clean["CONT_DOY"]).fillna(False).sum(),
        containment.isna().ne(fires_clean["CONT_DOY"].isna()).sum(),
        containment.dt.normalize().lt(discovery.dt.normalize()).fillna(False).sum(),
    ],
})
# Each date rule passes only when it has zero problem rows.
date_check["passed"] = date_check["problem_rows"].eq(0)
display(date_check)

,test,problem_rows,passed
0,Discovery year disagrees,0,True
1,Discovery day of year disagrees,0,True
2,Containment day of year disagrees,0,True
3,Containment date and day of year missing separ...,0,True
4,Containment date precedes discovery,0,True


## 3 Plan the dashboard — in class

Complete your plan before writing the app.

| Decision | Your plan |
| --- | --- |
| Audience and question | |
| Scope | |
| Two metrics and units | |
| Two filters | |
| Two charts and their purpose | |
| Missing values and empty results | |

Sketch the layout in your handout.

## 4 Build — in class

Create `dashboard.py` beside this notebook. Use Streamlit and Plotly to load
your saved analysis table and implement the class plan.

After creating it, run in Terminal or PowerShell:

```text
uv run streamlit run dashboard.py
```

Use the handout for the final review. Save a short progress note below.

## Progress note

- Completed:
- Unresolved:
- Next step:
